# SRM Valoração

## Imports

In [ ]:
import os
import sys
import time
from dataclasses import dataclass, field
from typing import Dict, List, Type, Optional, Tuple
import pandas as pd
import numpy as np

@dataclass
class MappingRule:
    """
    Define uma regra de composição de chaves para buscar valores em tabelas ZP.
    """
    target_column: str
    key_components: List[str]
    slice_limits: Optional[Dict[str, int]] = None

    def __init__(self, target_column: str, key_components: List[str], slice_limits: Optional[Dict[str, int]] = None):
        self.target_column = target_column
        self.key_components = key_components
        self.slice_limits = slice_limits


@dataclass
class ExcelSheetSchema:
    """
    Representa a definição estrutural (schema) de leitura e validação de uma planilha.
    """
    display_name: str
    header_row: int = 0
    engine: str = "openpyxl"
    dtypes: Dict[str, Type] = field(default_factory=dict)
    required_columns: List[str] = field(default_factory=list)
    mapping_rules: List[MappingRule] = field(default_factory=list)
    date_column_name: Optional[str] = None
    lookup_column_name: str = 'CHAVE'
    value_column_name: str = 'Cadastro'

    def __init__(
        self, 
        display_name: str, 
        header_row: int = 0, 
        engine: str = "openpyxl", 
        dtypes: Dict[str, Type] = None, 
        required_columns: List[str] = None,
        mapping_rules: List[MappingRule] = None,
        date_column_name: Optional[str] = None,
        lookup_column_name: str = 'CHAVE',
        value_column_name: str = 'Cadastro'
    ):
        self.display_name = display_name
        self.header_row = header_row
        self.engine = engine
        self.dtypes = dtypes if dtypes is not None else {}
        self.required_columns = required_columns if required_columns is not None else []
        self.mapping_rules = mapping_rules if mapping_rules is not None else []
        self.date_column_name = date_column_name
        self.lookup_column_name = lookup_column_name
        self.value_column_name = value_column_name

    def get_missing_columns(self, df_columns: List[str]) -> List[str]:
        """
        Retorna as colunas obrigatórias que estão ausentes na planilha real.
        """
        return [col for col in self.required_columns if col not in df_columns]




## Data Structures

In [84]:
# 1. Parâmetros de Execução
# (Colocamos aqui para facilitar a troca durante os testes)
periodo = 3
ano = 2026

# --- Schema da Planilha Principal (Ciclo N13) ---
CICLO_N13_SCHEMA = ExcelSheetSchema(
    display_name="Ciclo N13",
    header_row=3, 
    engine="calamine",
    dtypes={
        "EAN": str,
        "EAN Espelho": str,
        "COD_CLIENTE": str,
        "Company Code": str,
        "CD": str,
        "SKU": str
    },
    required_columns=[
        'Tipo 1', 'Tipo 2', 'Tipo 3', 'Regional', 'GP', 'Vend.', 'Gerente',
        'Rede', 'COD_CLIENTE', 'Company Code', 'CD', 'NOME_CLIENTE', 'UF', 
        'Região', 'EAN', 'SKU', 'Desc. SKU', 'Classificação', 'Tech', 'Tech 2',       
        'Subbrand', 'Size', 'Nivel 3 HieraR', 'Marca'
    ]
)

# --- Schema das Bases de Apoio (Clientes e Produtos) ---
BASE_CLIENTES_SCHEMA = ExcelSheetSchema(
    display_name="Base de Clientes",
    header_row=0,  
    engine="calamine",
    dtypes={
        "COD_CLIENTE": str,
        'COD REDE': str,
        'COD SUBREDE': str,
        'COND. PAG': str,
        'COD GP': str,

    },
    required_columns=[
        "COD_CLIENTE", 'COD REDE', 'COD SUBREDE', 'COND. PAG', 'COD GP'
    ]
)

BASE_PRODUTOS_SCHEMA = ExcelSheetSchema(
    display_name="Base de Produtos",
    header_row=0,  
    engine="calamine",
    dtypes={
        'EAN': str, 
        'Descrição': str,
        'SKU': str,
        'Family Price': str,
        'Ton/CDA': float, 
        'Unid/CX': float, 
        'Origem': str,
        'Hierarquia': str,
        'NCM': str,
        'kg/Un': float,
        'Class.': str,
        'H05': str,
        'LSV': float
    },
    required_columns=[
        "EAN", "SKU", "Descrição", "Family Price", "Class.",
        "Ton/CDA", "Unid/CX", "Origem", "Hierarquia", "NCM",
        "kg/Un", "H05", "LSV"
    ]
)



# --- Schemas das Tabelas ZP com Regras de Mapeamento ---

BASE_ZP55_SCHEMA = ExcelSheetSchema(
    display_name="Base de ZP55",
    header_row=1,
    engine="calamine",
    dtypes={
        'CHAVE': str,
        'Cadastro': float,
    },
    required_columns=[
        "CHAVE", "Cadastro"
    ],
    mapping_rules=[
        MappingRule(
            target_column = '55. CLIENTE',
            key_components = ['Company Code', 'COD_CLIENTE', 'Hierarquia']
        ),
        MappingRule(
            target_column = '55. CLIENTE H10',
            key_components = ['Company Code', 'COD_CLIENTE', 'Hierarquia'],
            slice_limits={"Hierarquia": 10}
        ), 
        MappingRule(
            target_column = '55. CD + UF DESTINO + Importação',
            key_components = ['CD', 'UF', 'Origem']
        ), 
        MappingRule(
            target_column = '55. CD + UF DESTINO + NCM',
            key_components = ['CD', 'UF', 'NCM']
        ), 
        MappingRule(
            target_column = '55. CD + UF DESTINO + H05',
            key_components = ['CD', 'UF', 'Hierarquia'],
            slice_limits={"Hierarquia": 10}
        ), 
    ]
)

BASE_ZP54_SCHEMA = ExcelSheetSchema(
    display_name="Base de ZP54",
    header_row=1,
    engine="calamine",
    dtypes={
        'CHAVE': str,
        'Cadastro': float,
    },
    required_columns=[
        "CHAVE", "Cadastro"
    ],
    mapping_rules=[
        MappingRule(
            target_column = '54. CLIENTE',
            key_components = ['Company Code', 'COD_CLIENTE', 'Hierarquia']
        ),
        MappingRule(
            target_column = '54. REDE',
            key_components = ['Company Code', 'COD SUBREDE', 'Hierarquia'],
        ), 
        MappingRule(
            target_column = '54. GP UF HIER 6',
            key_components = ['Company Code', 'CÓD GP', ' ','UF', 'Hierarquia'],
        ), 
        MappingRule(
            target_column = '54. GP UF HIER 5',
            key_components = ['Company Code', 'CÓD GP', ' ','UF', 'Hierarquia'],
            slice_limits={"Hierarquia": 10}
        ), 
    ]
)

BASE_ZP53_SCHEMA = ExcelSheetSchema(
    display_name="Base de ZP53",
    header_row=1,
    engine="calamine",
    dtypes={
        'CHAVE': str,
        'Cadastro': str,
        'P\'ANO_FIM': str,
    },
    required_columns=[
        "CHAVE", "Cadastro", "P\'ANO_FIM"
    ],
    date_column_name="P\'ANO_FIM",
    mapping_rules=[
        MappingRule(
            target_column = '53. EMISSOR',
            key_components = ['Company Code', 'COD_CLIENTE', 'Hierarquia']
        ),
        MappingRule(
            target_column = '53. REDE',
            key_components = ['Company Code', 'COD SUBREDE', 'Hierarquia'],
        ), 
        MappingRule(
            target_column = '53. GP UF',
            key_components = ['Company Code', 'CÓD GP', ' ', 'UF', 'Hierarquia'],
        ), 
        MappingRule(
            target_column = '53. GP',
            key_components = ['Company Code', 'CÓD GP', 'Hierarquia'],
            slice_limits={"Hierarquia": 10} 
        ), 
    ]
)

BASE_ZP52_SCHEMA = ExcelSheetSchema(
    display_name="Base de ZP52",
    header_row=1,
    engine="calamine",
    dtypes={
        'CHAVE': str,
        'Cadastro': str,
    },
    required_columns=[
        "CHAVE", "Cadastro"
    ],
    mapping_rules=[
        MappingRule(
            target_column = '52. H04',
            key_components = ['Company Code', 'COD_CLIENTE', 'Hierarquia'],
            slice_limits={"Hierarquia": 8}
        ),
        MappingRule(
            target_column = '52. H01',
            key_components = ['Company Code', 'COD SUBREDE', 'Hierarquia'],
            slice_limits={"Hierarquia": 2}
        )
    ]
)

BASE_ZP73_SCHEMA = ExcelSheetSchema(
    display_name="Base de ZP73",
    header_row=1,
    engine="calamine",
    dtypes={
        'CHAVE': str,
        'Cadastro': str,
    },
    required_columns=[
        "CHAVE", "Cadastro"
    ],
    mapping_rules=[
        MappingRule(
            target_column = '73. CD CLIENTE',
            key_components = ['Company Code', 'COD_CLIENTE'],
        ),
        MappingRule(
            target_column = '73. CD SUBREDE',
            key_components = ['Company Code', 'COD SUBREDE'],
        )
    ]
)

BASE_ZP70_SCHEMA = ExcelSheetSchema(
    display_name="Base de ZP70",
    header_row=0,
    engine="calamine",
    lookup_column_name='CONDICAO DE PAGAMENTO',
    value_column_name='Desconto',
    dtypes={
        'CONDICAO DE PAGAMENTO': str,
        'Desconto' : float,
    },
    required_columns=[
        'CONDICAO DE PAGAMENTO', 'Desconto'
    ],
    mapping_rules=[
        MappingRule(
            target_column = '70. COND. PAG',
            key_components = ['COND. PAG'],
        )
    ]
)

BASE_ZP39_SCHEMA = ExcelSheetSchema(
    display_name="Base de ZP39",
    header_row=1,
    engine="calamine",
    dtypes={
        'CHAVE': str,
        'Cadastro': str,
        'P\'ANO_FIM': str,
    },
    required_columns=[
        "CHAVE", "Cadastro", "P\'ANO_FIM"
    ],
    date_column_name="P\'ANO_FIM",
    mapping_rules=[
        MappingRule(
            target_column = '39. Emissor H12',
            key_components = ['Company Code', 'COD_CLIENTE', 'Hierarquia'],
        ),
        MappingRule(
            target_column = '39. Emissor H10',
            key_components = ['Company Code', 'COD_CLIENTE', 'Hierarquia'],
            slice_limits={"Hierarquia": 10}
        ),
        MappingRule(
            target_column = '39. Subrede H12',
            key_components = ['Company Code', 'COD SUBREDE', 'Hierarquia'],
        ),
        MappingRule(
            target_column = '39. GP UF H12',
            key_components = ['Company Code', 'CÓD GP', ' ', 'UF', 'Hierarquia'],
        ),
    ]
)

print("✅ Célula 2 executada: Todos os Schemas de dados foram definidos e estão prontos para uso.")


✅ Célula 2 executada: Todos os Schemas de dados foram definidos e estão prontos para uso.


## Montagem Ciclo N13

In [73]:
# Ciclo N13P
caminho_ciclo = f'../data/Ciclo_P{periodo:02d} N13P {ano} - envio.xlsx'

if not os.path.exists(caminho_ciclo):
    raise FileNotFoundError(f"❌ Erro crítico: Arquivo de ciclo não encontrado em {caminho_ciclo}")

colunas_base = CICLO_N13_SCHEMA.required_columns
colunas_periodos = [f'P{i:02d}-{ano}' for i in range(periodo, 14)]
colunas_a_ler = colunas_base + colunas_periodos

df_ciclo_n13 = pd.read_excel(
    caminho_ciclo,
    header=CICLO_N13_SCHEMA.header_row,
    usecols=colunas_a_ler,
    dtype=CICLO_N13_SCHEMA.dtypes,
    engine=CICLO_N13_SCHEMA.engine
    )

colunas_faltantes = CICLO_N13_SCHEMA.get_missing_columns(df_ciclo_n13.columns)
if colunas_faltantes:
    raise ValueError(f"❌ Erro: A planilha '{CICLO_N13_SCHEMA.display_name}' está fora do padrão! Colunas ausentes: {colunas_faltantes}")

df_ciclo_n13 = df_ciclo_n13.loc[:, ~df_ciclo_n13.columns.str.startswith('Unnamed:')]

# Clientes
caminho_clientes = '../data/BASE CLIENTES.xlsx'

if not os.path.exists(caminho_clientes):
    raise FileNotFoundError(f"❌ Erro crítico: Arquivo de clientes não encontrado em {caminho_clientes}")

df_clientes = pd.read_excel(
    caminho_clientes,
    header=BASE_CLIENTES_SCHEMA.header_row,
    dtype=BASE_CLIENTES_SCHEMA.dtypes,
    engine=BASE_CLIENTES_SCHEMA.engine
)

colunas_faltantes = BASE_CLIENTES_SCHEMA.get_missing_columns(df_clientes.columns)
if colunas_faltantes:
    print(colunas_faltantes)
    raise ValueError(f"❌ Erro: A planilha '{BASE_CLIENTES_SCHEMA.display_name}' está fora do padrão! Colunas ausentes: {colunas_faltantes}")


# Produtos
caminho_produtos = '../data/BASE PRODUTOS.xlsx'

if not os.path.exists(caminho_produtos):
    raise FileNotFoundError(f"❌ Erro crítico: Arquivo de produtos não encontrado em {caminho_produtos}")


df_produtos = pd.read_excel(
    caminho_produtos,
    header=BASE_PRODUTOS_SCHEMA.header_row,
    dtype=BASE_PRODUTOS_SCHEMA.dtypes,
    engine=BASE_PRODUTOS_SCHEMA.engine
)

df_produtos['kg/Un'] = df_produtos['kg/Un'].round(4)

colunas_faltantes = BASE_PRODUTOS_SCHEMA.get_missing_columns(df_produtos.columns)
if colunas_faltantes:
    print(colunas_faltantes)
    raise ValueError(f"❌ Erro: A planilha '{BASE_PRODUTOS_SCHEMA.display_name}' está fora do padrão! Colunas ausentes: {colunas_faltantes}")

# UF ORIGEM
dicionario_uf = {
    'BR01': 'SP',
    'BR03': 'PE',
    'BR30': 'SP',
    'BR31': 'MG',
}
df_ciclo_n13['UF ORIGEM'] = df_ciclo_n13['CD'].map(dicionario_uf)

# COD GP

dicionario_gp = {
    'ATACADO CASH & CARRY': 'AG',
    'GPA': 'AH',
    "SAM'S CLUB": 'AM',
    'GROCERY': 'AL',
    'ASSAI': 'AX',
    'Atacadão': 'TA',
    'DIST. MISTO': 'BI',
    'ESPECIALISTA DIRETO': 'AD',
    'DIA %': 'AF',
    'DIST. ALIMENTAR': 'AI',
    'DIST. ESPECIALISTA': 'AJ',
    'CENCOSUD': 'BJ',
    'CARREFOUR': 'AE',
    'ECOMMERCE': 'BO',
    'KA ESPECIALISTA': 'AQ',
    'PETZ': 'BL',
    'ATACADOS': 'AB',
    'MARTINS': 'BK',
    'COBASI': 'BM',
    'ATACADOS ESPECIAIS': 'BT',
    'CONVENIENCIAS': 'BP'
}
df_ciclo_n13['CÓD GP'] = df_ciclo_n13['GP'].map(dicionario_gp)
df_ciclo_n13['GP'] = df_ciclo_n13['GP'].str.strip()

# EAN Espelho
df_produtos_limpo = df_produtos.drop_duplicates(subset=['EAN'], keep='first')
df_produtos_desc_limpo = df_produtos.drop_duplicates(subset=['Descrição'], keep='first')

set_ean_validos = set(df_produtos_limpo['EAN'])

dic_desc_para_ean = df_produtos_desc_limpo.set_index('Descrição')['EAN'].to_dict()

df_ciclo_n13['EAN Espelho'] = np.where(
    df_ciclo_n13['EAN'].isin(set_ean_validos),
    df_ciclo_n13['EAN'],
    np.nan
)

busca_secundaria = df_ciclo_n13['Desc. SKU'].map(dic_desc_para_ean)

df_ciclo_n13['EAN Espelho'] = df_ciclo_n13['EAN Espelho'].fillna(busca_secundaria)
df_ciclo_n13['EAN Espelho'] = df_ciclo_n13['EAN Espelho'].astype(str)

# SKU 
colunas_produtos = BASE_PRODUTOS_SCHEMA.required_columns
df_prod_exato = df_produtos[colunas_produtos].drop_duplicates(subset=['EAN', 'SKU'], keep='first')
df_prod_resgate = df_produtos[colunas_produtos].drop(columns=['SKU']).drop_duplicates(subset=['EAN'], keep='first')

df_ciclo_n13 = pd.merge(
    df_ciclo_n13, df_prod_exato,
    left_on=['EAN Espelho', 'SKU'], right_on=['EAN', 'SKU'],
    how='left', suffixes=('', '_exato')
)
df_ciclo_n13 = df_ciclo_n13.drop(columns=['EAN_exato'], errors='ignore')

df_ciclo_n13 = pd.merge(
    df_ciclo_n13, df_prod_resgate,
    left_on='EAN Espelho', right_on='EAN',
    how='left', suffixes=('', '_resgate')
)
colunas_preencher_prod = [col for col in colunas_produtos if col not in ['EAN', 'SKU']]
for col in colunas_preencher_prod:
    df_ciclo_n13[col] = df_ciclo_n13[col].fillna(df_ciclo_n13[f'{col}_resgate'])

colunas_lixo_prod = [f'{col}_resgate' for col in colunas_preencher_prod] + ['EAN_resgate']
df_ciclo_n13 = df_ciclo_n13.drop(columns=colunas_lixo_prod, errors='ignore')

colunas_clientes = BASE_CLIENTES_SCHEMA.required_columns
 
df_cli_exato = df_clientes[colunas_clientes].drop_duplicates(subset=['COD_CLIENTE'], keep='first')

df_ciclo_n13 = pd.merge(
    df_ciclo_n13,
    df_cli_exato,
    on='COD_CLIENTE',
    how='left'
)

## Calculos

In [78]:
def processar_zp_dinamico(df_principal: pd.DataFrame, schema: ExcelSheetSchema, caminho_arquivo: str, nome_coluna_final: str) -> pd.DataFrame:
    """
    Função mestre para carregar, processar e aplicar os valores de qualquer planilha ZP.
    
    Esta função é genérica e utiliza as regras do Schema para:
    1. Ler a planilha ZP.
    2. Criar dicionários de busca para valores e, opcionalmente, para datas.
    3. Gerar chaves compostas dinamicamente.
    4. Mapear os valores e datas para o DataFrame principal.
    5. Resolver a hierarquia de fallbacks (`fillna`).
    
    Args:
        df_principal (pd.DataFrame): O DataFrame principal (df_ciclo_n13) a ser modificado.
        schema (ExcelSheetSchema): O objeto de schema da ZP a ser processada.
        caminho_arquivo (str): O caminho para o arquivo .xlsx da ZP.
        nome_coluna_final (str): O nome da coluna final que conterá o valor consolidado (ex: 'ZP55').
        
    Returns:
        pd.DataFrame: O DataFrame principal modificado com as novas colunas da ZP.
    """
    print(f"⚙️ Iniciando processamento dinâmico para: {schema.display_name}...")
    start_time = time.time()
    
    # --- 1. LEITURA E PREPARAÇÃO DA ZP ---
    df_zp = pd.read_excel(caminho_arquivo, header=schema.header_row, dtype=schema.dtypes, engine='calamine', usecols=schema.required_columns)
    
    coluna_lookup = schema.lookup_column_name
    coluna_valor = schema.value_column_name

    df_zp[coluna_lookup] = df_zp[coluna_lookup].astype(str).str.strip()
    df_zp[coluna_valor] = pd.to_numeric(df_zp[coluna_valor], errors='coerce').fillna(0.0)


    # Dicionário de busca de valores (obrigatório)
    dic_valores = df_zp.drop_duplicates(subset=[coluna_lookup], keep='first').set_index(coluna_lookup)[coluna_valor].to_dict()
    
    # Dicionário de busca de datas (opcional, só se o schema declarar)
    dic_datas = None
    if schema.date_column_name:
        print(f"   -> Coluna de data '{schema.date_column_name}' detectada. Criando dicionário de datas...")
        df_zp[schema.date_column_name] = df_zp[schema.date_column_name].astype(str).str.strip()
        dic_datas = df_zp.drop_duplicates(subset=[coluna_lookup], keep='first').set_index(coluna_lookup)[schema.date_column_name].to_dict()

    # --- 2. GERAÇÃO DINÂMICA DE CHAVES E MAPEAMENTO ---
    colunas_valores_calculadas = []
    colunas_datas_calculadas = []

    for rule in schema.mapping_rules:
        key_series_list = []
        for col in rule.key_components:
            if col in df_principal.columns:
                serie = df_principal[col].astype(str).str.strip()
                if rule.slice_limits and col in rule.slice_limits:
                    serie = serie.str[:rule.slice_limits[col]]
            else:
                serie = pd.Series([col] * len(df_principal), index=df_principal.index)
            key_series_list.append(serie)
        
        # Concatenação inteligente de chaves
        chave_composta = key_series_list[0]
        for i in range(1, len(key_series_list)):
            if rule.key_components[i] == " " or rule.key_components[i-1] == " ":
                chave_composta += key_series_list[i]
            else:
                chave_composta += '_' + key_series_list[i]
        
        # Mapeamento de Valores
        df_principal[rule.target_column] = (chave_composta.map(dic_valores).fillna(0.0) / 100)
        colunas_valores_calculadas.append(rule.target_column)

        # Mapeamento de Datas (se aplicável)
        if dic_datas:
            # Adiciona um prefixo 'd.' para a coluna de data correspondente
            coluna_data_target = f"d.{rule.target_column}"
            df_principal[coluna_data_target] = chave_composta.map(dic_datas)
            colunas_datas_calculadas.append(coluna_data_target)

    # --- 3. RESOLUÇÃO HIERÁRQUICA (FALLBACK) - VERSÃO CORRIGIDA ---
    # Para o valor final (ex: 'ZP55', 'ZP53')
    
    # Começamos com a primeira coluna da hierarquia
    df_principal[nome_coluna_final] = df_principal[colunas_valores_calculadas[0]]

    # Loop inteligente com np.where: se o valor atual for 0, busca na próxima coluna da hierarquia
    for col in colunas_valores_calculadas[1:]:
        df_principal[nome_coluna_final] = np.where(
            df_principal[nome_coluna_final] == 0.0,      # A Condição: Se o valor atual for zero...
            df_principal[col],                           # Valor se VERDADEIRO: ...pegue o valor da próxima coluna...
            df_principal[nome_coluna_final]              # Valor se FALSO: ...senão, mantenha o valor que já estava.
        )

    # Aplica o arredondamento final de 4 casas
    df_principal[nome_coluna_final] = df_principal[nome_coluna_final].fillna(0.0).round(4)
    
    # Para a data final (se aplicável) - a lógica de fillna aqui está correta
    if colunas_datas_calculadas:
        nome_coluna_data_final = f"{nome_coluna_final}_data"
        df_principal[nome_coluna_data_final] = df_principal[colunas_datas_calculadas[0]]
        for col in colunas_datas_calculadas[1:]:
            df_principal[nome_coluna_data_final] = df_principal[nome_coluna_data_final].fillna(df_principal[col])

    end_time = time.time()
    print(f"✅ {schema.display_name} processado com sucesso em {end_time - start_time:.2f} segundos.")
    
    return df_principal





### ZP55 e ZP54

In [79]:
# Agora, em vez de repetir o código, você simplesmente CHAMA a função para cada ZP!
df_ciclo_n13 = processar_zp_dinamico(df_ciclo_n13, BASE_ZP55_SCHEMA, '../data/ZP55.xlsx', 'ZP55')
df_ciclo_n13 = processar_zp_dinamico(df_ciclo_n13, BASE_ZP54_SCHEMA, '../data/ZP54.xlsx', 'ZP54')

# df_ciclo_n13.head(50)

⚙️ Iniciando processamento dinâmico para: Base de ZP55...
✅ Base de ZP55 processado com sucesso em 4.45 segundos.
⚙️ Iniciando processamento dinâmico para: Base de ZP54...
✅ Base de ZP54 processado com sucesso em 16.32 segundos.


### GSV

In [80]:
print("💰 Calculando métricas de GSV e projeções futuras...")
inicio_gsv_proj = time.time()

colunas_para_converter = ['LSV', 'ZP55', 'ZP54', 'kg/Un', 'Unid/CX']
for col in colunas_para_converter:
    df_ciclo_n13[col] = pd.to_numeric(df_ciclo_n13[col], errors='coerce').fillna(0)

df_ciclo_n13['GSV/CDA'] = (
    df_ciclo_n13['LSV'] *
    (1 + df_ciclo_n13['ZP55']) *
    (1 + df_ciclo_n13['ZP54'])
).round(4)

denominador = df_ciclo_n13['kg/Un'] * df_ciclo_n13['Unid/CX']

df_ciclo_n13['GSV/TON'] = np.where(
    denominador == 0,
    0,
    (df_ciclo_n13['GSV/CDA'] / denominador) * 1000
).round(4)

colunas_periodos = [f'P{i:02d}-{ano}' for i in range(periodo, 14)]

print(f"   -> Gerando projeções para os períodos: {colunas_periodos}")
for p_col in colunas_periodos:
    
    df_ciclo_n13[p_col] = pd.to_numeric(df_ciclo_n13[p_col], errors='coerce').fillna(0)
    
    nome_coluna_projecao = f'GSV R$ {p_col}'
    df_ciclo_n13[nome_coluna_projecao] = (
        df_ciclo_n13[p_col] * df_ciclo_n13['GSV/TON']
    ).round(24)

tempo_gsv_proj = time.time() - inicio_gsv_proj
print(f"✅ Cálculos de GSV e projeções concluídos em {tempo_gsv_proj:.2f} segundos.")

colunas_para_exibir = ['LSV', 'ZP55', 'ZP54', 'GSV/CDA', 'GSV/TON'] + [f'GSV R$ {colunas_periodos[1]}']
df_ciclo_n13[colunas_para_exibir].head()


💰 Calculando métricas de GSV e projeções futuras...
   -> Gerando projeções para os períodos: ['P03-2026', 'P04-2026', 'P05-2026', 'P06-2026', 'P07-2026', 'P08-2026', 'P09-2026', 'P10-2026', 'P11-2026', 'P12-2026', 'P13-2026']
✅ Cálculos de GSV e projeções concluídos em 0.14 segundos.


,LSV,ZP55,ZP54,GSV/CDA,GSV/TON,GSV R$ P04-2026
0,199.0,0.4826,-0.5039,146.3681,16263.1222,-0.451960
1,199.0,0.4826,-0.5039,146.3681,16263.1222,-8.040284
2,299.4,0.4826,-0.5655,192.8704,11905.5802,-174.126769
3,199.0,0.4826,-0.5654,128.2233,14247.0333,-31.675232
4,299.4,0.0867,-0.5130,158.4493,9780.8210,-443.734066


### ZP53, ZP52, ZP73, ZP70 e ZP39

In [85]:
df_ciclo_n13 = processar_zp_dinamico(df_ciclo_n13, BASE_ZP53_SCHEMA, '../data/ZP53.xlsx', 'ZP53')
df_ciclo_n13 = processar_zp_dinamico(df_ciclo_n13, BASE_ZP52_SCHEMA, '../data/ZP52.xlsx', 'ZP52')
df_ciclo_n13 = processar_zp_dinamico(df_ciclo_n13, BASE_ZP73_SCHEMA, '../data/ZP73.xlsx', 'ZP73')
df_ciclo_n13 = processar_zp_dinamico(df_ciclo_n13, BASE_ZP70_SCHEMA, '../data/ZP70.xlsx', 'ZP70')
df_ciclo_n13 = processar_zp_dinamico(df_ciclo_n13, BASE_ZP39_SCHEMA, '../data/ZP39.xlsx', 'ZP39')

⚙️ Iniciando processamento dinâmico para: Base de ZP53...
   -> Coluna de data 'P'ANO_FIM' detectada. Criando dicionário de datas...
✅ Base de ZP53 processado com sucesso em 5.06 segundos.
⚙️ Iniciando processamento dinâmico para: Base de ZP52...
✅ Base de ZP52 processado com sucesso em 1.66 segundos.
⚙️ Iniciando processamento dinâmico para: Base de ZP73...
✅ Base de ZP73 processado com sucesso em 0.86 segundos.
⚙️ Iniciando processamento dinâmico para: Base de ZP70...
✅ Base de ZP70 processado com sucesso em 0.12 segundos.
⚙️ Iniciando processamento dinâmico para: Base de ZP39...
   -> Coluna de data 'P'ANO_FIM' detectada. Criando dicionário de datas...
✅ Base de ZP39 processado com sucesso em 3.41 segundos.


### NIV

In [88]:
print("🧾 Calculando métricas de NIV...")
inicio_niv = time.time()

colunas_para_converter_niv = [
    'GSV/CDA', 'ZP53', 'ZP52', 'ZP73', 'ZP70', 'ZP39',
    'kg/Un', 'Unid/CX'
]

for col in colunas_para_converter_niv:
    if col in df_ciclo_n13.columns:
        df_ciclo_n13[col] = pd.to_numeric(df_ciclo_n13[col], errors='coerce').fillna(0)
    else:
        print(f"   -> Alerta: Coluna '{col}' não encontrada para o cálculo de NIV. Será considerada como 0.")
        df_ciclo_n13[col] = 0

df_ciclo_n13['NIV/CDA'] = (
    df_ciclo_n13['GSV/CDA'] *
    (1 + df_ciclo_n13['ZP53']) *
    (1 + df_ciclo_n13['ZP52']) *
    (1 + df_ciclo_n13['ZP73']) *
    (1 + df_ciclo_n13['ZP70']) *
    (1 + df_ciclo_n13['ZP39'])
).round(4)


denominador_niv = df_ciclo_n13['kg/Un'] * df_ciclo_n13['Unid/CX']

df_ciclo_n13['NIV/TON'] = np.where(
    denominador_niv == 0,
    0, 
    (df_ciclo_n13['NIV/CDA'] / denominador_niv) * 1000
).round(4)


tempo_niv = time.time() - inicio_niv
print(f"✅ Cálculos de NIV concluídos em {tempo_niv:.2f} segundos.")


df_ciclo_n13[['GSV/CDA', 'ZP53', 'ZP52', 'NIV/CDA', 'NIV/TON']].head()


🧾 Calculando métricas de NIV...
✅ Cálculos de NIV concluídos em 0.06 segundos.


,GSV/CDA,ZP53,ZP52,NIV/CDA,NIV/TON
0,146.3681,0.0000,0.00,146.3681,16263.1222
1,146.3681,0.0000,0.00,146.3681,16263.1222
2,192.8704,0.0000,-0.11,171.7576,10602.3210
3,128.2233,0.0000,-0.11,114.1872,12687.4667
4,158.4493,-0.0729,0.00,146.9130,9068.7037


In [90]:
df_ciclo_n13.columns.tolist()

['Tipo 1',
 'Tipo 2',
 'Tipo 3',
 'Regional',
 'GP',
 'Vend.',
 'Gerente',
 'Rede',
 'COD_CLIENTE',
 'Company Code',
 'CD',
 'NOME_CLIENTE',
 'UF',
 'Região',
 'EAN',
 'SKU',
 'Desc. SKU',
 'Classificação',
 'Tech',
 'Tech 2',
 'Subbrand',
 'Size',
 'Nivel 3 HieraR',
 'Marca',
 'P03-2026',
 'P04-2026',
 'P05-2026',
 'P06-2026',
 'P07-2026',
 'P08-2026',
 'P09-2026',
 'P10-2026',
 'P11-2026',
 'P12-2026',
 'P13-2026',
 'UF ORIGEM',
 'CÓD GP',
 'EAN Espelho',
 'Descrição',
 'Family Price',
 'Class.',
 'Ton/CDA',
 'Unid/CX',
 'Origem',
 'Hierarquia',
 'NCM',
 'kg/Un',
 'H05',
 'LSV',
 'COD REDE',
 'COD SUBREDE',
 'COND. PAG',
 'COD GP',
 '55. CLIENTE',
 '55. CLIENTE H10',
 '55. CD + UF DESTINO + Importação',
 '55. CD + UF DESTINO + NCM',
 '55. CD + UF DESTINO + H05',
 'ZP55',
 '54. CLIENTE',
 '54. REDE',
 '54. GP UF HIER 6',
 '54. GP UF HIER 5',
 'ZP54',
 'GSV/CDA',
 'GSV/TON',
 'GSV R$ P03-2026',
 'GSV R$ P04-2026',
 'GSV R$ P05-2026',
 'GSV R$ P06-2026',
 'GSV R$ P07-2026',
 'GSV R$ P08

## Exportação